In [0]:
from pyspark.sql.functions import count, max, min, avg, sum, round

In [0]:
# Load the enriched trip dataset
df = spark.read.table("nyctaxi.02_silver.yellow_trips_enriched")

In [0]:
#df.display()

In [0]:
# Aggregate trip data by pickup date with key metrics

df = df.groupBy(df.tpep_pickup_datetime.cast("date").alias("pickup_date") ).agg(
            count("*").alias("total_trips"),                             # total number of trips per day
            round(avg("passenger_count"), 1).alias("average_passengers"), # average passengers per trip
            round(avg("trip_distance"), 1).alias("average_distance"),     # average trip distance (miles)
            round(avg("fare_amount"), 2).alias("average_fare_per_trip"),   # average fare per trip ($)
            max("fare_amount").alias("max_fare"),                         # highest single-trip fare
            min("fare_amount").alias("min_fare"),                         # lowest single-trip fare
            round(sum("total_amount"), 2).alias("total_revenue")          # total revenue for the day ($)
        )

In [0]:
#display(df)

pickup_date,total_trips,average_passengers,average_distance,average_fare_per_trip,max_fare,min_fare,total_revenue
2025-06-07,163816,1.4,6.2,19.35,655.0,-655.0,4404477.45
2025-07-24,134564,1.3,5.3,18.84,1984.7,-537.8,3701486.95
2025-07-25,137005,1.3,5.0,18.22,2495.0,-629.5,3610404.01
2025-06-05,177340,1.3,4.7,19.06,624.06,-483.2,4906698.38
2025-06-26,152851,1.3,4.7,19.78,2588.1,-263.4,4372703.07
2025-07-11,124211,1.3,6.6,18.17,600.0,-600.0,3295826.82
2025-07-20,123945,1.4,7.4,18.96,642.8,-599.0,3315734.2
2025-07-22,123086,1.3,9.8,18.47,450.0,-339.0,3344436.15
2025-06-15,125074,1.4,8.3,18.73,3237.36,-900.0,3318256.43
2025-06-23,141713,1.3,7.4,19.25,1625.0,-800.0,3957929.28


In [0]:
# Write the daily summary to a Unity Catalog managed Delta table in the gold schema
df.write.mode("overwrite").saveAsTable("nyctaxi.03_gold.daily_trip_summary")